In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# 1. Load Processed Data Artifacts
data_path = "artifacts/features/processed_data.npz"
print(f"Loading data from: {data_path}...")
data = np.load(data_path)

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val     = data['X_val'], data['y_val']
X_test, y_test   = data['X_test'], data['y_test']

print(f"Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}")

# 2. Baseline Model (Dummy Classifier - Most Frequent)
print("\n--- 1. Evaluating Baseline Model ---")
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)

# Calculate Dummy Metrics
dummy_val_probs = np.full(len(y_val), y_train.mean())
print(f"Baseline (Train Prior) PR-AUC: {average_precision_score(y_val, dummy_val_probs):.4f}")

# 3. Fast Linear Model (Logistic Regression with fast solver)
print("\n--- 2. Training Logistic Regression ---")
lr_model = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

lr_val_probs = lr_model.predict_proba(X_val)[:, 1]
print(f"Logistic Regression Val ROC-AUC: {roc_auc_score(y_val, lr_val_probs):.4f}")
print(f"Logistic Regression Val PR-AUC : {average_precision_score(y_val, lr_val_probs):.4f}")

# 4. Tree-based Model (Fast HistGradientBoosting)
print("\n--- 3. Training HistGradientBoosting ---")
hgb_model = HistGradientBoostingClassifier(
    class_weight='balanced',
    max_iter=50,
    max_leaf_nodes=15,
    random_state=42
)
hgb_model.fit(X_train, y_train)

hgb_val_probs = hgb_model.predict_proba(X_val)[:, 1]
print(f"Gradient Boosting Val ROC-AUC: {roc_auc_score(y_val, hgb_val_probs):.4f}")
print(f"Gradient Boosting Val PR-AUC : {average_precision_score(y_val, hgb_val_probs):.4f}")

# 5. Final Evaluation on Test Set (Using HistGradientBoosting)
print("\n=======================================================")
print("--- FINAL EVALUATION ON UNSEEN TEST SET ---")
best_model = hgb_model
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)[:, 1]

print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probs):.4f}")
print(f"Test PR-AUC : {average_precision_score(y_test, test_probs):.4f}")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, test_preds, target_names=['On-Time (0)', 'Late (1)']))
print("=======================================================")

# 6. Save Model 
os.makedirs("artifacts/models", exist_ok=True)
model_path = "artifacts/models/best_model.joblib"
joblib.dump(best_model, model_path)
print(f"\nBest model artifact successfully saved to: {model_path}")

Loading data from: artifacts/features/processed_data.npz...
Train: 77176 | Val: 9647 | Test: 9647

--- 1. Evaluating Baseline Model ---
Baseline (Train Prior) PR-AUC: 0.0201

--- 2. Training Logistic Regression ---
Logistic Regression Val ROC-AUC: 0.7499
Logistic Regression Val PR-AUC : 0.0595

--- 3. Training HistGradientBoosting ---
Gradient Boosting Val ROC-AUC: 0.7327
Gradient Boosting Val PR-AUC : 0.0622

--- FINAL EVALUATION ON UNSEEN TEST SET ---
Test ROC-AUC: 0.4496
Test PR-AUC : 0.0757

Classification Report (Test Set):
              precision    recall  f1-score   support

 On-Time (0)       0.90      0.42      0.57      8820
    Late (1)       0.07      0.50      0.13       827

    accuracy                           0.43      9647
   macro avg       0.49      0.46      0.35      9647
weighted avg       0.83      0.43      0.53      9647


Best model artifact successfully saved to: artifacts/models/best_model.joblib
